In [2]:
from utile_all_with_vol_ind import *

import pandas as pd
pd.set_option('future.no_silent_downcasting', True)
import numpy as np
import yfinance as yf

portfolio_path = 'data/portfolio_holdings.csv'
benchmark = 'QQQ'
train_start = '2003-01-01'
train_end = '2017-12-31'
test_start = '2018-01-01'
test_end = '2024-12-31'

# Load portfolio tickers
tickers = load_portfolio(portfolio_path)

# Include benchmark for comparison
if benchmark.upper() not in tickers:
    tickers.append(benchmark.upper())

# Download price data
prices, volume = download_price_data(tickers, train_start, test_end)
prices.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/portfolio_holdings.csv'

In [11]:
# Compute daily log returns
log_rets = compute_log_returns(prices)
# Compute volume indicators
volume_indicators = compute_volume_indicators(volume, prices)
volume_indicators.head()


,vol_trend_RDDT,vol_trend_NVDA,vol_trend_SMR,vol_trend_MU,vol_trend_MRVL,vol_trend_MSFT,vol_trend_ASML,vol_trend_AEM,vol_trend_AMD,vol_trend_VERU,...,vol_momentum_AMD,vol_momentum_VERU,vol_momentum_AI,vol_momentum_GOOGL,vol_momentum_INGM,vol_momentum_PLUG,vol_momentum_IONQ,vol_momentum_RGTI,vol_momentum_ARBE,vol_momentum_QQQ
Date,,,,,,,,,,,,,,,,,,,,,
2003-01-31,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0,0,0,0,0,0,0,0,0,0
2003-02-28,-0.095238,-0.097130,-0.095238,-0.277958,-0.002397,-0.216268,-0.541625,-0.254158,-0.535026,-0.570841,...,-0.25506,-0.41081,0.0,0.0,0.0,-0.446357,0.0,0.0,0.0,-0.076547
2003-03-31,0.105263,-0.152544,0.105263,0.240368,0.166515,0.018510,0.881986,0.034879,0.630047,1.226054,...,0.238045,0.429769,0.0,0.0,0.0,-0.199266,0.0,0.0,0.0,0.115765
2003-04-30,0.000000,-0.301337,0.000000,-0.262943,-0.212567,-0.089135,-0.250993,0.155117,0.419814,-0.653471,...,0.165212,-0.38363,0.0,0.0,0.0,-0.009966,0.0,0.0,0.0,-0.09189
2003-05-31,0.000000,1.964017,0.000000,0.526052,0.413441,0.049124,0.464770,-0.089201,-0.365457,1.983444,...,-0.250325,0.545362,0.0,0.0,0.0,0.566401,0.0,0.0,0.0,0.046749


In [12]:
monthly_log_returns = np.log(prices.resample('ME').last()).diff().dropna()
# Compute monthly indicators
monthly_indicators = compute_monthly_indicators(prices, log_rets, volume)
# Check the results
print(f"Monthly indicators shape: {monthly_indicators.shape}")
print(f"Monthly returns shape: {monthly_log_returns.shape}")

monthly_indicators.head()

Monthly indicators shape: (263, 369)
Monthly returns shape: (263, 18)


,"(vol, RDDT)","(vol, NVDA)","(vol, SMR)","(vol, MU)","(vol, MRVL)","(vol, MSFT)","(vol, ASML)","(vol, AEM)","(vol, AMD)","(vol, VERU)",...,vol_momentum_AMD,vol_momentum_VERU,vol_momentum_AI,vol_momentum_GOOGL,vol_momentum_INGM,vol_momentum_PLUG,vol_momentum_IONQ,vol_momentum_RGTI,vol_momentum_ARBE,vol_momentum_QQQ
Date,,,,,,,,,,,,,,,,,,,,,
2003-02-28,0.0,1.273688,0.0,0.533284,0.523353,0.298252,0.458301,0.519209,0.453572,0.491943,...,-0.25506,-0.41081,0.0,0.0,0.0,-0.446357,0.0,0.0,0.0,-0.076547
2003-03-31,0.0,1.235304,0.0,0.735661,0.525173,0.364748,0.718866,0.550585,0.654056,0.416738,...,0.238045,0.429769,0.0,0.0,0.0,-0.199266,0.0,0.0,0.0,0.115765
2003-04-30,0.0,0.752890,0.0,0.509735,0.546536,0.323105,0.610552,0.680574,0.534722,0.456300,...,0.165212,-0.38363,0.0,0.0,0.0,-0.009966,0.0,0.0,0.0,-0.09189
2003-05-31,0.0,1.120735,0.0,0.410065,0.583454,0.238475,0.635125,0.442980,0.346565,0.514080,...,-0.250325,0.545362,0.0,0.0,0.0,0.566401,0.0,0.0,0.0,0.046749
2003-06-30,0.0,0.658979,0.0,0.455919,0.423420,0.268291,0.468148,0.354181,0.530135,0.423474,...,0.195165,-0.42378,0.0,0.0,0.0,-0.231459,0.0,0.0,0.0,0.049554


In [ ]:

# Fetch sentiment data
yahoo_sent = fetch_yahoo_sentiment(tickers, start=full_start, end=full_end)
reddit_sent = fetch_reddit_sentiment(tickers, start=full_start, end=full_end)
sentiment = aggregate_sentiment(yahoo_sent, reddit_sent)

# Align sentiment with indicators
sentiment = sentiment.loc[indicators.index]

# Compute correlation between sentiment and future returns (t+1)
future_returns = shift_returns_for_validation(monthly_pct_returns, periods=1)
# Align index
future_returns = future_returns.loc[indicators.index]
sentiment_corr = compute_sentiment_correlation(sentiment, future_returns)
print("Sentiment vs. next-month return correlation:")
print(sentiment_corr)

# Normalize features using training period
normalized_indicators, scaler = normalize_features(
    indicators, training_period=(train_start[:7], train_end[:7])
)
# Combine normalized indicators and sentiment as final state representation
# We'll keep sentiment unnormalized to preserve interpretability
final_features = pd.concat([normalized_indicators, sentiment.add_prefix('sent_')], axis=1)
# Persist final features to disk for later RL training
final_features.to_csv('monthly_features_with_sentiment.csv')
print("Generated monthly features with sentiment saved to 'monthly_features_with_sentiment.csv'.")

